In [1]:
from pathlib import Path
import nibabel as nib
import pandas as pd
import numpy as np

In [2]:
img = nib.load("data/sub-AF488_FLAIR.nii.gz")
print("Dimensiones:", img.shape)

n_voxeles = np.prod(img.shape)
print("Cantidad total de vóxeles:", n_voxeles)

Dimensiones: (176, 256, 256)
Cantidad total de vóxeles: 11534336


In [3]:
rows = []

root = Path("H:/descarga_20052026/")

for subdir in root.glob("sub-*"):

    anat = subdir / "anat"

    if not anat.exists():
        continue

    flair_files = list(anat.glob("*_FLAIR.nii.gz"))

    if len(flair_files) == 0:
        continue

    flair_file = flair_files[0]

    try:
        img = nib.load(str(flair_file))

        rows.append({
            "subject": subdir.name,
            "shape": img.shape,
            "voxel_size": img.header.get_zooms()[:3]
        })

    except Exception as e:
        rows.append({
            "subject": subdir.name,
            "shape": None,
            "voxel_size": None
        })

df = pd.DataFrame(rows)

print(df.head())
print(f"Sujetos con FLAIR: {len(df)}")

     subject            shape       voxel_size
0  sub-AF023  (176, 256, 256)  (1.0, 1.0, 1.0)
1  sub-AF024  (176, 256, 256)  (1.0, 1.0, 1.0)
2  sub-AF025  (176, 256, 256)  (1.0, 1.0, 1.0)
3  sub-AF026  (176, 256, 256)  (1.0, 1.0, 1.0)
4  sub-AF027  (176, 256, 256)  (1.0, 1.0, 1.0)
Sujetos con FLAIR: 2084


In [5]:
df["shape"].value_counts()

shape
(192, 256, 256)    592
(176, 256, 256)    308
(160, 256, 256)    276
(321, 240, 240)    166
(160, 512, 512)    146
(180, 256, 256)    135
(512, 512, 248)    129
(179, 256, 256)    100
(192, 224, 224)     78
(288, 288, 162)     54
(256, 256, 179)     17
(512, 512, 252)     10
(512, 512, 256)      8
(192, 240, 256)      6
(512, 512, 280)      5
(180, 288, 288)      5
(240, 240, 240)      4
(512, 512, 268)      3
(512, 512, 160)      2
(512, 512, 284)      2
(512, 512, 264)      2
(192, 512, 512)      2
(224, 256, 256)      2
(190, 256, 256)      2
(176, 240, 256)      2
(256, 256, 160)      2
(192, 384, 383)      2
(512, 512, 272)      1
(512, 512, 240)      1
(512, 512, 308)      1
(512, 512, 332)      1
(512, 512, 276)      1
(512, 512, 304)      1
(512, 512, 292)      1
(512, 512, 260)      1
(512, 512, 288)      1
(340, 320, 75)       1
(135, 256, 256)      1
(256, 256, 152)      1
(137, 256, 256)      1
(208, 256, 256)      1
(186, 256, 256)      1
(180, 240, 240)      1
(200,

In [6]:
df["voxel_size"].value_counts()

voxel_size
(1.0, 0.9765625, 0.9765625)    742
(1.0, 1.0, 1.0)                566
(0.6, 1.0416666, 1.0416666)    165
(0.4688, 0.4688, 0.5)          163
(1.0, 0.49, 0.49)              115
                              ... 
(0.99986607, 1.0, 1.0)           1
(0.9994795, 1.0, 1.0)            1
(0.9998438, 1.0, 1.0)            1
(0.6875, 0.6875, 5.0)            1
(0.6, 1.0419576, 1.0419576)      1
Name: count, Length: 77, dtype: int64

In [7]:
from pathlib import Path

n_flair = 0
n_t1 = 0
n_both = 0

for subdir in root.glob("sub-*"):

    anat = subdir / "anat"

    if not anat.exists():
        continue

    has_flair = len(list(anat.glob("*_FLAIR.nii.gz"))) > 0
    has_t1 = len(list(anat.glob("*_T1w.nii.gz"))) > 0

    if has_flair:
        n_flair += 1

    if has_t1:
        n_t1 += 1

    if has_flair and has_t1:
        n_both += 1

print("FLAIR:", n_flair)
print("T1:", n_t1)
print("Ambas:", n_both)

FLAIR: 2084
T1: 3136
Ambas: 2079


In [8]:
from pathlib import Path

n_multi_flair = 0
n_multi_t1 = 0

for subdir in root.glob("sub-*"):

    anat = subdir / "anat"

    if not anat.exists():
        continue

    n_flair = len(list(anat.glob("*FLAIR.nii.gz")))
    n_t1 = len(list(anat.glob("*T1w.nii.gz")))

    if n_flair > 1:
        n_multi_flair += 1

    if n_t1 > 1:
        n_multi_t1 += 1

print("Sujetos con >1 FLAIR:", n_multi_flair)
print("Sujetos con >1 T1:", n_multi_t1)

Sujetos con >1 FLAIR: 32
Sujetos con >1 T1: 126


In [9]:
img = nib.load("data/ples_lpa_maf488_sub-AF488_FLAIR.nii")
data = img.get_fdata()

print("shape:", data.shape)
print("NaN:", np.isnan(data).sum())
print("Finito:", np.isfinite(data).sum())
print("No cero:", np.count_nonzero(np.nan_to_num(data)))

shape: (176, 256, 256)
NaN: 188
Finito: 11534148
No cero: 3093


In [10]:
vals = data[np.isfinite(data)]
vals = vals[vals > 0]

print("min:", vals.min())
print("max:", vals.max())
print("n valores únicos:", len(np.unique(vals)))

min: 0.041236381977796555
max: 0.9982784986495972
n valores únicos: 3093


In [11]:
from collections import Counter

# voxel_size es la columna que ya tienes en df

max_res = [max(vs) for vs in df["voxel_size"]]

print("Total sujetos:", len(max_res))
print("<= 0.5 mm :", sum(r <= 0.5 for r in max_res))
print("<= 0.7 mm :", sum(r <= 0.7 for r in max_res))
print("<= 1.2 mm :", sum(r <= 1.2 for r in max_res))
print("> 2.0 mm  :", sum(r > 2.0 for r in max_res))
print("> 3.0 mm  :", sum(r > 3.0 for r in max_res))
print("> 4.0 mm  :", sum(r > 4.0 for r in max_res))

Total sujetos: 2084
<= 0.5 mm : 165
<= 0.7 mm : 168
<= 1.2 mm : 2082
> 2.0 mm  : 1
> 3.0 mm  : 1
> 4.0 mm  : 1


In [12]:
from pathlib import Path
from collections import defaultdict

root = Path(r"H:\descarga_20052026")

counts = defaultdict(lambda: {"T1": 0, "FLAIR": 0, "BOTH": 0})

for subdir in root.glob("sub-*"):

    subject = subdir.name

    # centro = sub-XX (dos letras después de sub-)
    center = subject.split("-")[1][:2]

    anat = subdir / "anat"
    if not anat.exists():
        continue

    has_t1 = len(list(anat.glob("*_T1w.nii.gz"))) > 0
    has_flair = len(list(anat.glob("*_FLAIR.nii.gz"))) > 0

    if has_t1:
        counts[center]["T1"] += 1
    if has_flair:
        counts[center]["FLAIR"] += 1
    if has_t1 and has_flair:
        counts[center]["BOTH"] += 1

for c, v in sorted(counts.items()):
    print(c, v)

AF {'T1': 298, 'FLAIR': 146, 'BOTH': 146}
BE {'T1': 114, 'FLAIR': 102, 'BOTH': 102}
BN {'T1': 201, 'FLAIR': 170, 'BOTH': 170}
CU {'T1': 488, 'FLAIR': 444, 'BOTH': 441}
L0 {'T1': 1, 'FLAIR': 0, 'BOTH': 0}
LO {'T1': 179, 'FLAIR': 124, 'BOTH': 124}
MA {'T1': 217, 'FLAIR': 216, 'BOTH': 216}
MI {'T1': 474, 'FLAIR': 454, 'BOTH': 454}
PA {'T1': 2, 'FLAIR': 0, 'BOTH': 0}
PB {'T1': 6, 'FLAIR': 0, 'BOTH': 0}
PC {'T1': 109, 'FLAIR': 0, 'BOTH': 0}
PI {'T1': 156, 'FLAIR': 0, 'BOTH': 0}
PM {'T1': 147, 'FLAIR': 0, 'BOTH': 0}
PS {'T1': 248, 'FLAIR': 0, 'BOTH': 0}
RE {'T1': 52, 'FLAIR': 48, 'BOTH': 48}
SL {'T1': 212, 'FLAIR': 209, 'BOTH': 209}
TA {'T1': 217, 'FLAIR': 166, 'BOTH': 164}
cu {'T1': 15, 'FLAIR': 5, 'BOTH': 5}


In [13]:
import deepinv as dinv
print(dinv.__version__)

d:\AgustinaV\repos\wmhpatterns\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


0.4.1


In [14]:
import inspect
from deepinv.physics.noise import NoiseModel
from deepinv.optim import DataFidelity

print(inspect.signature(NoiseModel))
print(inspect.signature(DataFidelity))

(noise_model: 'Callable' = None, rng: 'torch.Generator' = None)
(d: 'Callable[[torch.Tensor, torch.Tensor], torch.Tensor]' = None)


In [15]:
print(dir(DataFidelity))

['T_destination', '__annotate_func__', '__call__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_apply', '_call_impl', '_compiled_call_impl', '_get_backward_hooks', '_get_backward_pre_hooks', '_get_name', '_load_from_state_dict', '_maybe_warn_non_full_backward_hook', '_named_members', '_register_load_state_dict_pre_hook', '_register_state_dict_hook', '_replicate_for_data_parallel', '_save_to_state_dict', '_slow_forward', '_version', '_wrapped_call_impl', 'add', 'add_module', 'apply', 'bfloat16', 'bregman_prox', 'buffers', 'call_super_init', 'children', 'compile', 'conjugate', 'cpu', 'cuda', 'double', '